# ARQWELIA Lot 2 — SDXL Inpainting on a free GPU (benchmark only)

**WARNING**
- GPU availability is NOT guaranteed on free environments.
- A free environment is NOT appropriate for Production.
- Delete temporary files after the session.
- Do NOT use a real user photo during Phase 0A (synthetic benchmark images only).
- Do NOT expose ComfyUI on the Internet; no public tunnel is included.
- Do NOT store any DeepSeek API key in this notebook.
- The image + mask stay local; nothing is sent to a remote provider.
- This notebook is the FIRST free execution path (official Diffusers pipeline).
  The ComfyUI checkpoint is NOT resolved/installed for this benchmark and its
  preflight BLOCKS until a verified checkpoint exists.

## 1. Verify CUDA GPU presence

In [ ]:
import subprocess
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True, timeout=20)
    print(out.stdout or "NO GPU / nvidia-smi not found")
except Exception as exc:
    print("GPU check failed:", exc)

## 2. Configuration

`MODEL_REVISION` is PINNED (immutable) — verified before the official run.
Never silently use `revision="main"`.

- repo: `diffusers/stable-diffusion-xl-1.0-inpainting-0.1`
- revision: `115134f363124c53c7d878647567d04daf26e41e`
- license: OpenRAIL++

In [ ]:
MODEL_ID = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
MODEL_REVISION = "115134f363124c53c7d878647567d04daf26e41e"
SOURCE_PATH = "dataset/photos/synthetic01.png"
MASK_PATH = "dataset/masks/synthetic01-pool-mask.png"
WORKING = 1024
OUT_DIR = "benchmark-out/deepseek-comfyui-poc"

# --- SECOND benchmark (POC experimental, prepared but NOT executed here) ---
# First run visually failed: mask regenerated as grass, no pool; prompt was
# truncated (95 tokens > CLIP 77). These params are experimental POC values.
POC_PARAMS = {
    "seed": 43,
    "num_inference_steps": 35,
    "guidance_scale": 8.0,
    "strength": 0.99,
    "padding_mask_crop": 64,
    "note": "POC experimental — tune after a manual visual check",
}

# --- quality gate (manual visual acceptance) ---
generationStatus = "pending"        # succeeded | failed
visualAcceptance = "pending"        # pending | accepted | rejected
rejectionReason = None              # e.g. no_pool_visible_grass_regenerated

print("MODEL_ID:", MODEL_ID)
print("MODEL_REVISION:", MODEL_REVISION)
print("POC_PARAMS:", POC_PARAMS)
assert MODEL_REVISION is not None and len(MODEL_REVISION) > 0, "MODEL_REVISION must be set (pinned) before the official run"


## 3. Install locked dependencies

In [ ]:
import sys
!{sys.executable} -m pip install --quiet torch==2.4.1 diffusers==0.31.0 transformers==4.44.2 accelerate==0.34.2 pillow==10.4.0
print("dependencies installed")

## 4. Load SDXL Inpainting (official Diffusers pipeline)

In [ ]:
import torch
from diffusers import StableDiffusionXLInpaintPipeline

# GPU gate: only CUDA availability is required. A GPU named "Tesla T4" is
# accepted. We never require the literal "NVIDIA" string.
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required for this benchmark")

pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)

# GPU memory mode = model_cpu_offload (do NOT combine with pipe.to("cuda")).
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

gpu = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
print("GPU:", gpu)
print("VRAM total (GB):", round(props.total_memory / 1024**3, 2))
print("memory mode: model_cpu_offload")
print("revision:", MODEL_REVISION)
print("dtype:", pipe.dtype)


## 5. Load source + mask and prepare the 1024x1024 working canvas

- proportional resize (fit inside, never stretched);
- centered padding to 1024x1024;
- the SAME scale + offsets applied to the grayscale mask (nearest);
- mapping recorded.

In [ ]:
from PIL import Image

img = Image.open(SOURCE_PATH).convert("RGB")
mask = Image.open(MASK_PATH).convert("L")
orig_w, orig_h = img.size
assert mask.size == img.size, "mask and image must be the same size"

scale = min(WORKING / orig_w, WORKING / orig_h)
resized_w = max(1, round(orig_w * scale))
resized_h = max(1, round(orig_h * scale))
offset_x = max(0, (WORKING - resized_w) // 2)
offset_y = max(0, (WORKING - resized_h) // 2)

working_image = Image.new("RGB", (WORKING, WORKING), (0, 0, 0))
working_image.paste(img.resize((resized_w, resized_h)), (offset_x, offset_y))
working_mask = Image.new("L", (WORKING, WORKING), 0)
working_mask.paste(mask.resize((resized_w, resized_h), Image.NEAREST), (offset_x, offset_y))

mapping = {
    "scale": scale,
    "offsetX": offset_x,
    "offsetY": offset_y,
    "resizedWidth": resized_w,
    "resizedHeight": resized_h,
    "originalWidth": orig_w,
    "originalHeight": orig_h,
    "workingWidth": WORKING,
    "workingHeight": WORKING,
}
print("working image size:", working_image.size, "| mask size:", working_mask.size)
print("mapping:", mapping)

## 6. Use a VisualBrief JSON

In [ ]:
import json

# CLIP-safe prompts (<= 75 tokens). The first run truncated a 95-token prompt;
# we now enforce the limit with the pipeline tokenizer (fail closed).
POSITIVE_PROMPT = ("Large photorealistic rectangular in-ground swimming pool, "
                   "clear blue water, natural limestone coping, correctly embedded "
                   "in this lawn, realistic perspective and sunlight, Mediterranean "
                   "residential garden.")
NEGATIVE_PROMPT = ("grass inside pool, empty lawn, pond, people, text, logo, "
                   "distorted house, warped geometry, extra pool, artificial "
                   "reflections.")

def _clip_tokens(text):
    try:
        return len(pipe.tokenizer(text, return_tensors="pt", padding=False, truncation=False).input_ids[0])
    except Exception:
        return len(text.split())  # conservative fallback

# Fail closed: reject if either prompt exceeds the CLIP 77 limit (target <= 75).
positive_tokens = _clip_tokens(POSITIVE_PROMPT)
negative_tokens = _clip_tokens(NEGATIVE_PROMPT)
assert positive_tokens <= 75, f"Positive prompt too long: {positive_tokens} tokens > 75"
assert negative_tokens <= 75, f"Negative prompt too long: {negative_tokens} tokens > 75"

visual_brief = {
    "version": "arqwelia-visual-brief-v1",
    "concept": "A",
    "sceneType": "residential_garden_pool_inpainting",
    "pool": {"shape": "rectangular", "estimatedDimensions": "8x4m", "placement": "central_open_lawn", "orientation": "parallel_to_house"},
    "preserve": ["house_architecture", "camera_perspective", "boundary_fences", "mature_trees", "unmasked_pixels"],
    "add": ["realistic_in_ground_pool", "natural_stone_coping", "mediterranean_landscaping"],
    "negative": ["people", "text", "logo", "house_distortion", "extra_buildings", "duplicate_pool", "floating_objects", "unrealistic_reflections"],
    "inpaintingPrompt": POSITIVE_PROMPT,
    "negativePrompt": NEGATIVE_PROMPT,
    "recommended": {"steps": POC_PARAMS["num_inference_steps"], "cfg": POC_PARAMS["guidance_scale"], "strength": POC_PARAMS["strength"], "seed": POC_PARAMS["seed"]},
}
print("visual brief ready:", visual_brief["version"], visual_brief["concept"])
print("positive tokens:", positive_tokens, "| negative tokens:", negative_tokens)


## 7. Run ONE manual generation with the working canvas

In [ ]:
import torch

generationStatus = "in_progress"
generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(visual_brief["recommended"]["seed"])
result = pipe(
    prompt=visual_brief["inpaintingPrompt"],
    negative_prompt=visual_brief["negativePrompt"],
    image=working_image,
    mask_image=working_mask,
    width=WORKING,
    height=WORKING,
    num_inference_steps=visual_brief["recommended"]["steps"],
    guidance_scale=visual_brief["recommended"]["cfg"],
    strength=visual_brief["recommended"]["strength"],
    padding_mask_crop=POC_PARAMS["padding_mask_crop"],
    generator=generator,
)
generationStatus = "succeeded"
print("generation done")


## 8. Recompute on the working canvas (masked = generated, unmasked = source)

In [ ]:
generated = result.images[0].convert("RGB")
canvas_composite = Image.composite(generated, working_image, working_mask)
print("canvas composite size:", canvas_composite.size)

## 9. Restore to the ORIGINAL aspect ratio (no black padding bands)

Crop the useful area out of the canvas, resize to original dims, recompose on
the ORIGINAL source outside the mask. Final output = 1536x1024 for synthetic01.

In [ ]:
from PIL import ImageChops

# crop useful area from canvas
crop_box = (offset_x, offset_y, offset_x + resized_w, offset_y + resized_h)
cropped = canvas_composite.crop(crop_box).resize((orig_w, orig_h), Image.LANCZOS)

# resize the ORIGINAL mask to original dims (nearest)
mask_orig = mask.resize((orig_w, orig_h), Image.NEAREST)

# composite generated area onto the ORIGINAL source
final_output = Image.composite(cropped, img, mask_orig)
print("final output size:", final_output.size)

## 10. Save PNG + mapping + SHA-256

In [ ]:
import hashlib, os
os.makedirs(OUT_DIR, exist_ok=True)

def file_sha256(path):
    """SHA-256 of the actual PNG file bytes."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def pixel_sha256(img):
    """SHA-256 of the RAW PIXELS only (NOT the file bytes). Keep separate."""
    return hashlib.sha256(img.tobytes()).hexdigest()

# Pixel-level quality metrics: changed pixels INSIDE the mask, unchanged pixels
# OUTSIDE the mask (relative to the working canvas source).
import numpy as np
src_arr = np.asarray(working_image.convert("RGB")).astype(int)
gen_arr = np.asarray(generated.convert("RGB")).astype(int)
mask_arr = np.asarray(working_mask.convert("L")) >= 128
diff = np.abs(src_arr - gen_arr).sum(axis=2) > 12
inside_mask = mask_arr
outside_mask = ~mask_arr
changed_inside = (diff & inside_mask).sum() / max(inside_mask.sum(), 1)
unchanged_outside = (~diff & outside_mask).sum() / max(outside_mask.sum(), 1)

final_path = os.path.join(OUT_DIR, "notebook-sdxl-final.png")
canvas_composite.save(os.path.join(OUT_DIR, "notebook-sdxl-canvas.png"))
final_output.save(final_path)
with open(os.path.join(OUT_DIR, "notebook-mapping.json"), "w") as f:
    json.dump(mapping, f, indent=2)

finalFileSha256 = file_sha256(final_path)
finalPixelSha256 = pixel_sha256(final_output)
print("saved")
print("working canvas pixel sha256:", pixel_sha256(canvas_composite))
print("final FILE sha256 (finalFileSha256):", finalFileSha256)
print("final PIXEL sha256 (finalPixelSha256):", finalPixelSha256)


## 11. Print technical metadata (measured output dims)

In [ ]:
print("seed:", visual_brief["recommended"]["seed"])
print("steps:", visual_brief["recommended"]["steps"])
print("cfg:", visual_brief["recommended"]["cfg"])
print("strength:", visual_brief["recommended"]["strength"])
print("padding_mask_crop:", POC_PARAMS["padding_mask_crop"])
print("working canvas:", canvas_composite.size)
print("final output size:", final_output.size)
print("model:", MODEL_ID, "| revision:", MODEL_REVISION)
print("changedPixelRatioInsideMask:", round(changed_inside, 4))
print("unchangedPixelRatioOutsideMask:", round(unchanged_outside, 4))
print("promptPositiveTokenCount:", positive_tokens)
print("promptNegativeTokenCount:", negative_tokens)
print("effectiveInferenceSteps:", visual_brief["recommended"]["steps"])
print("generationStatus:", generationStatus)
print("visualAcceptance:", visualAcceptance)
print("rejectionReason:", rejectionReason)


## 12. Stop

Generation complete. Delete temporary files and free the GPU after the session.